# NASA C-MAPSS Turbofan — Remaining Useful Life (RUL) Prediction

**Goal:** Predict how many cycles an aircraft engine has left before failure.

**Pipeline:**
1. Load Dataset → 2. Calculate RUL → 3. Remove Constant Sensors → 4. Normalize →
5. Save → 6. Rolling Features → 7. Train Models → 8. Evaluate →
9. SHAP Explanations → 10. Feature Pruning → 11. Compare Results

## 1 · Setup — Locate the Dataset

In [ ]:
from pathlib import Path
import zipfile
import os

# Folder where we will extract and save all data
EXTRACT_DIR = Path("/kaggle/working/cmapss")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Search for the zip file under the Kaggle input or working folders
zip_file = next(
    (p for p in Path("/kaggle/input").rglob("CMAPSSData.zip")), None
) or next(
    (p for p in Path("/kaggle/working").glob("CMAPSSData.zip")), None
)

if zip_file:
    print(f"Found zip file: {zip_file}")
    print("Extracting...")
    with zipfile.ZipFile(zip_file) as z:
        z.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    # Kaggle already extracted the dataset automatically
    EXTRACT_DIR = Path("/kaggle/input")
    print("Zip not found — assuming Kaggle already extracted the dataset.")
    print(f"Looking for files under: {EXTRACT_DIR}")

In [ ]:
# Find the 6 required data files (FD001 and FD003 each have train / test / RUL)
REQUIRED_FILES = [
    "train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt",
    "train_FD003.txt", "test_FD003.txt", "RUL_FD003.txt",
]

found_files = {}

for folder, subfolders, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:
        if filename in REQUIRED_FILES:
            found_files[filename] = Path(folder) / filename

missing_files = [f for f in REQUIRED_FILES if f not in found_files]
if missing_files:
    raise FileNotFoundError(f"These files were not found: {missing_files}")

print("All required files found:")
for filename, filepath in found_files.items():
    print(f"  {filename:25s}  →  {filepath}")

## 2 · Imports and Constants

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Folder to save processed (cleaned) data
PROCESSED_DIR = Path("/kaggle/working/cmapss/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Column names for the raw data files (no header in the original files)
ALL_COLUMNS = (
    ["unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

# Engines are considered healthy above this RUL value
# so we cap RUL at 125 to avoid large outliers
RUL_CAP = 125

DATASETS = ["FD001", "FD003"]

print("Columns:", ALL_COLUMNS)
print(f"RUL cap: {RUL_CAP}")
print(f"Datasets: {DATASETS}")

## 3 · Load, Label, Clean, and Normalize

In [ ]:
def load_raw_data(dataset_name):
    """
    Load the train, test, and ground-truth RUL files for one dataset (e.g. FD001).
    Returns three DataFrames: training data, test data, and final RUL values.
    """

    def read_txt_file(file_key):
        return pd.read_csv(
            found_files[f"{file_key}_{dataset_name}.txt"],
            sep=r"\s+",        # columns are separated by whitespace
            header=None,        # no header row in the raw files
            names=ALL_COLUMNS
        )

    training_data  = read_txt_file("train")
    test_data      = read_txt_file("test")

    # RUL file only has one column — the remaining life at the last test cycle
    final_rul = pd.read_csv(
        found_files[f"RUL_{dataset_name}.txt"],
        sep=r"\s+",
        header=None,
        names=["RUL"]
    )

    return training_data, test_data, final_rul

In [ ]:
def calculate_train_rul(training_data):
    """
    Calculate Remaining Useful Life (RUL) for every row in the training set.

    How it works:
      - Find the last (maximum) cycle number for each engine.
      - RUL = last cycle - current cycle  (engine counting down to failure).
      - Cap RUL at RUL_CAP so the model focuses on the degradation zone.
    """
    training_data = training_data.copy()

    # The last cycle for each engine (looked up per-row using transform)
    last_cycle_per_engine = training_data.groupby("unit_id")["cycle"].transform("max")

    # Remaining Useful Life = Final cycle − Current cycle
    raw_rul = last_cycle_per_engine - training_data["cycle"]

    # Cap at RUL_CAP so very healthy engines don't inflate the target
    training_data["RUL"] = np.minimum(raw_rul, RUL_CAP)

    return training_data

In [ ]:
def calculate_test_rul(test_data, final_rul):
    """
    Calculate RUL for every row in the test set.

    The RUL file gives each engine's remaining life AT its last observed cycle.
    So: Total Life = last observed cycle + remaining life given in RUL file.
    Then for any row: RUL = Total Life − current cycle  (capped at RUL_CAP).
    """
    test_data  = test_data.copy()
    final_rul  = final_rul.copy()

    # Assign engine IDs (1, 2, 3 ...) to the RUL file rows
    final_rul["unit_id"] = range(1, len(final_rul) + 1)

    # Find the last recorded cycle for each test engine
    last_observed_cycle = (
        test_data.groupby("unit_id")["cycle"]
        .max()
        .reset_index()
        .rename(columns={"cycle": "last_cycle"})
    )

    # Attach the ground-truth final RUL to each engine
    engine_info = last_observed_cycle.merge(final_rul, on="unit_id")

    # Total engine life = last observed cycle + RUL remaining at that point
    engine_info["total_life"] = engine_info["last_cycle"] + engine_info["RUL"]

    # Bring total_life into the test dataframe
    test_data = test_data.merge(engine_info[["unit_id", "total_life"]], on="unit_id")

    # RUL at each row = Total life − current cycle  (capped)
    test_data["RUL"] = np.minimum(test_data["total_life"] - test_data["cycle"], RUL_CAP)

    # Remove the helper column
    test_data = test_data.drop(columns="total_life")

    return test_data

In [ ]:
def remove_constant_sensors(training_data, test_data, variance_threshold=1e-5):
    """
    Remove sensors whose values barely change across the training set.
    A sensor with near-zero standard deviation carries no useful signal.
    We use training data statistics only — test data is never used here.
    """
    sensor_columns = [
        col for col in training_data.columns
        if col not in ("unit_id", "cycle", "RUL")
    ]

    # Calculate how much each sensor varies in the training data
    sensor_std = training_data[sensor_columns].std()

    # Keep only sensors with meaningful variation
    useful_sensors  = sensor_std[sensor_std > variance_threshold].index.tolist()
    removed_sensors = [s for s in sensor_columns if s not in useful_sensors]

    print(f"Removed {len(removed_sensors)} constant sensors: {removed_sensors}")
    print(f"Kept {len(useful_sensors)} useful sensors.")

    # Rebuild both DataFrames with only the useful columns
    keep_columns = ["unit_id", "cycle"] + useful_sensors + ["RUL"]
    training_data = training_data[keep_columns]
    test_data     = test_data[keep_columns]

    return training_data, test_data, useful_sensors

In [ ]:
def normalize_sensors(training_data, test_data, sensor_columns):
    """
    Scale all sensor values to the range [0, 1] using Min-Max Scaling.

    Formula:  scaled = (value − min) / (max − min)

    Important: min and max are calculated from TRAINING data only.
    The same values are then applied to the test data.
    This prevents data leakage — we never peek at test data during preprocessing.
    """
    training_normalized = training_data.copy()
    test_normalized     = test_data.copy()

    sensor_min = training_data[sensor_columns].min()
    sensor_max = training_data[sensor_columns].max()

    # Range for each sensor (replace 0 with 1 to avoid division by zero)
    sensor_range = (sensor_max - sensor_min).replace(0, 1)

    # Apply the same scaling to both training and test
    training_normalized[sensor_columns] = (training_data[sensor_columns] - sensor_min) / sensor_range
    test_normalized[sensor_columns]     = (test_data[sensor_columns]     - sensor_min) / sensor_range

    return training_normalized, test_normalized

In [ ]:
def prepare_dataset(dataset_name):
    """
    Full preprocessing pipeline for one C-MAPSS dataset.
    Runs all steps in order and saves the cleaned files.
    """
    print(f"\n{'='*55}")
    print(f"  Preparing dataset: {dataset_name}")
    print(f"{'='*55}")

    # ── Step 1: Load raw data ─────────────────────────────────
    training_data, test_data, final_rul = load_raw_data(dataset_name)
    print(f"Raw training rows : {len(training_data)}")
    print(f"Raw test rows     : {len(test_data)}")

    # ── Step 2: Calculate RUL labels ─────────────────────────
    training_data = calculate_train_rul(training_data)
    test_data     = calculate_test_rul(test_data, final_rul)
    print(f"RUL range (train) : {training_data['RUL'].min()} – {training_data['RUL'].max()}")
    print(f"RUL mean  (train) : {training_data['RUL'].mean():.1f}")

    # ── Step 3: Remove constant sensors ──────────────────────
    training_data, test_data, useful_sensors = remove_constant_sensors(training_data, test_data)

    # ── Step 4: Normalize sensor values ──────────────────────
    training_data, test_data = normalize_sensors(training_data, test_data, useful_sensors)
    print("Normalization applied (Min-Max, using training statistics only).")

    # ── Step 5: Save processed files ─────────────────────────
    training_data.to_csv(PROCESSED_DIR / f"processed_{dataset_name}_train.csv", index=False)
    test_data.to_csv(    PROCESSED_DIR / f"processed_{dataset_name}_test.csv",  index=False)
    print(f"Saved processed files to: {PROCESSED_DIR}")

    return training_data, test_data, useful_sensors


# Run the pipeline for both datasets
preprocessing_results = {}
for dataset_name in DATASETS:
    preprocessing_results[dataset_name] = prepare_dataset(dataset_name)

In [ ]:
# Quick sanity check — confirm shapes look right
for dataset_name in DATASETS:
    training_data, test_data, useful_sensors = preprocessing_results[dataset_name]
    print(f"{dataset_name}  →  train shape: {training_data.shape},  test shape: {test_data.shape}")

# Preview the first few rows of FD001 training data
preprocessing_results["FD001"][0].head()

## 4 · Rolling-Window Feature Engineering

In [ ]:
ROLLING_WINDOW = 5   # look back 5 cycles when computing rolling statistics

def add_rolling_features(data, sensor_columns, window=ROLLING_WINDOW):
    """
    For each sensor, add two new features:
      - Rolling Mean : average sensor value over the last `window` cycles.
      - Rolling Std  : how much the sensor fluctuated over those cycles.

    These features help the model detect degradation trends, not just snapshots.
    Calculated per engine so cycles never bleed across different engines.
    min_periods=1 means we still get a value even for the very first cycle.
    """
    # Sort so each engine's cycles are in order
    data = data.sort_values(["unit_id", "cycle"]).reset_index(drop=True)

    grouped = data.groupby("unit_id")[sensor_columns]

    # Rolling mean — smoothed signal over the last `window` cycles
    rolling_mean = (
        grouped.rolling(window, min_periods=1)
        .mean()
        .reset_index(drop=True)
    )
    rolling_mean.columns = [f"{sensor}_rollmean" for sensor in sensor_columns]

    # Rolling std — variability over the last `window` cycles
    rolling_std = (
        grouped.rolling(window, min_periods=1)
        .std()
        .fillna(0)   # first cycle has no std yet, fill with 0
        .reset_index(drop=True)
    )
    rolling_std.columns = [f"{sensor}_rollstd" for sensor in sensor_columns]

    # Combine original data with the new rolling features
    data_with_features = pd.concat([data, rolling_mean, rolling_std], axis=1)

    return data_with_features

In [ ]:
def create_engineered_features(dataset_name):
    """
    Load the preprocessed data, add rolling features, and save the result.
    """
    print(f"\n{'='*55}")
    print(f"  Feature engineering: {dataset_name}")
    print(f"{'='*55}")

    # Load cleaned data
    training_data = pd.read_csv(PROCESSED_DIR / f"processed_{dataset_name}_train.csv")
    test_data     = pd.read_csv(PROCESSED_DIR / f"processed_{dataset_name}_test.csv")

    # Identify sensor columns (exclude id/cycle/target)
    sensor_columns = [
        col for col in training_data.columns
        if col not in ("unit_id", "cycle", "RUL")
    ]

    print(f"Original sensor count : {len(sensor_columns)}")

    # Add rolling mean and std for each sensor
    training_engineered = add_rolling_features(training_data, sensor_columns)
    test_engineered     = add_rolling_features(test_data,     sensor_columns)

    # Save
    training_engineered.to_csv(PROCESSED_DIR / f"features_{dataset_name}_train.csv", index=False)
    test_engineered.to_csv(    PROCESSED_DIR / f"features_{dataset_name}_test.csv",  index=False)

    total_features = training_engineered.shape[1] - 3  # exclude unit_id, cycle, RUL
    print(f"Final feature count   : {total_features}  "
          f"({len(sensor_columns)} raw + {len(sensor_columns)} rollmean + {len(sensor_columns)} rollstd)")
    print(f"Training shape: {training_engineered.shape}")
    print(f"Test shape    : {test_engineered.shape}")

    return training_engineered, test_engineered


feature_results = {}
for dataset_name in DATASETS:
    feature_results[dataset_name] = create_engineered_features(dataset_name)

## 5 · Model Training and Stacking Ensemble

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Folder to save trained model files
MODEL_DIR = Path("/kaggle/working/cmapss/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

NUMBER_OF_FOLDS = 3    # number of cross-validation folds
RANDOM_SEED     = 42   # for reproducibility

def get_base_models():
    """
    Return the 4 base learners used in the stacking ensemble.
    Each model is created fresh to avoid state leaking between runs.
    """
    return {
        "Random Forest": RandomForestRegressor(
            n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.08,
            random_state=RANDOM_SEED, n_jobs=-1, verbosity=0
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.08,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        ),
        "CatBoost": CatBoostRegressor(
            iterations=100, depth=5, learning_rate=0.08,
            random_state=RANDOM_SEED, verbose=0, thread_count=-1
        ),
    }

print("Base models ready:", list(get_base_models().keys()))

In [ ]:
def nasa_score(true_rul, predicted_rul):
    """
    NASA asymmetric scoring function used in the PHM08 challenge.

    Calculates d = predicted RUL − true RUL for each engine, then:
      - d < 0 : early prediction (predicted failure sooner than reality) — less penalised
      - d > 0 : late prediction  (predicted failure later than reality)  — more penalised

    A lower score is better. The asymmetry reflects that late predictions
    are more dangerous in real maintenance scenarios.
    """
    difference = predicted_rul - true_rul

    score = np.where(
        difference < 0,
        np.exp(-difference / 13) - 1,   # early prediction penalty
        np.exp( difference / 10) - 1    # late prediction penalty (steeper)
    )

    return np.sum(score)

In [ ]:
def train_all_models(dataset_name):
    """
    Train 4 base models and combine them with a Ridge stacking ensemble.

    Training strategy:
      1. Use GroupKFold cross-validation (grouped by engine) to generate
         out-of-fold (OOF) predictions for the training data.
         GroupKFold ensures all cycles from one engine stay in the same fold,
         so the model never sees future cycles of an engine it is predicting on.
      2. Train a final version of each model on ALL training data.
      3. Pass the base model predictions into a Ridge meta-model (stacking).
    """
    print(f"\n{'='*55}")
    print(f"  Training models for: {dataset_name}")
    print(f"{'='*55}")

    # Load feature-engineered data
    training_data = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_train.csv")
    test_data     = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_test.csv")

    # Identify feature columns
    feature_columns = [
        col for col in training_data.columns
        if col not in ("unit_id", "cycle", "RUL")
    ]

    X_train     = training_data[feature_columns]
    y_train     = training_data["RUL"]
    engine_ids  = training_data["unit_id"]   # used to keep engines together in CV

    # For test evaluation, only use the LAST observed cycle of each engine
    # (this is the cycle where we want to predict remaining life)
    last_cycle_index = test_data.groupby("unit_id")["cycle"].idxmax()
    X_test_final     = test_data.loc[last_cycle_index, feature_columns]
    y_test_final     = test_data.loc[last_cycle_index, "RUL"]

    print(f"Training samples : {len(X_train)}")
    print(f"Training engines : {engine_ids.nunique()}")
    print(f"Test engines     : {len(X_test_final)}")
    print(f"Feature count    : {len(feature_columns)}")

    # Storage for predictions (filled in during cross-validation)
    oof_predictions  = {name: np.zeros(len(X_train))      for name in get_base_models()}
    test_predictions = {name: np.zeros(len(X_test_final)) for name in get_base_models()}

    group_kfold = GroupKFold(n_splits=NUMBER_OF_FOLDS)

    # ── Train each base model ─────────────────────────────────────────────────
    for model_name in get_base_models():
        print(f"\n── {model_name}")

        # Step 1: Generate out-of-fold predictions via cross-validation
        for fold_number, (train_index, val_index) in enumerate(
            group_kfold.split(X_train, y_train, engine_ids), start=1
        ):
            fold_model = get_base_models()[model_name]
            fold_model.fit(X_train.iloc[train_index], y_train.iloc[train_index])
            oof_predictions[model_name][val_index] = fold_model.predict(X_train.iloc[val_index])
            print(f"   Fold {fold_number}/{NUMBER_OF_FOLDS} complete")

        # Step 2: Train final model on ALL training data
        final_model = get_base_models()[model_name]
        final_model.fit(X_train, y_train)
        test_predictions[model_name] = final_model.predict(X_test_final)

        # Save the trained model
        joblib.dump(final_model, MODEL_DIR / f"{dataset_name}_{model_name}.joblib")

        # Report performance
        oof_rmse  = np.sqrt(mean_squared_error(y_train,      oof_predictions[model_name]))
        test_rmse = np.sqrt(mean_squared_error(y_test_final, test_predictions[model_name]))
        test_nasa = nasa_score(y_test_final.values, test_predictions[model_name])
        print(f"   OOF RMSE: {oof_rmse:.2f}  |  Test RMSE: {test_rmse:.2f}  |  NASA Score: {test_nasa:.0f}")

    # ── Build Stacking Ensemble ───────────────────────────────────────────────
    print(f"\n── Stacking Ensemble (Ridge meta-model)")

    # The meta-model takes each base model's prediction as its input
    meta_train_input = np.column_stack([oof_predictions[n]  for n in get_base_models()])
    meta_test_input  = np.column_stack([test_predictions[n] for n in get_base_models()])

    meta_model = Ridge(alpha=1.0, positive=True)
    meta_model.fit(meta_train_input, y_train)
    stacking_predictions = meta_model.predict(meta_test_input)

    stacking_rmse  = np.sqrt(mean_squared_error(y_test_final, stacking_predictions))
    stacking_nasa  = nasa_score(y_test_final.values, stacking_predictions)
    model_weights  = dict(zip(get_base_models(), np.round(meta_model.coef_, 3)))

    joblib.dump(meta_model, MODEL_DIR / f"{dataset_name}_meta_ridge.joblib")

    print(f"   Test RMSE : {stacking_rmse:.2f}")
    print(f"   NASA Score: {stacking_nasa:.0f}")
    print(f"   Weights   : {model_weights}")

    # Collect all results
    return {
        "dataset":           dataset_name,
        "base_rmse":         {n: np.sqrt(mean_squared_error(y_test_final, test_predictions[n])) for n in get_base_models()},
        "base_nasa":         {n: nasa_score(y_test_final.values, test_predictions[n])            for n in get_base_models()},
        "stacking_rmse":     stacking_rmse,
        "stacking_nasa":     stacking_nasa,
        "model_weights":     model_weights,
        "oof_predictions":   oof_predictions,
        "test_predictions":  test_predictions,
        "stacking_pred":     stacking_predictions,
        "y_test":            y_test_final,
        "X_test":            X_test_final,
        "feature_columns":   feature_columns,
    }


training_results = {}
for dataset_name in DATASETS:
    training_results[dataset_name] = train_all_models(dataset_name)

In [ ]:
# Build a summary table comparing all models across both datasets
summary_rows = []

for dataset_name, results in training_results.items():
    for model_name in results["base_rmse"]:
        summary_rows.append({
            "Dataset":    dataset_name,
            "Model":      model_name,
            "RMSE":       round(results["base_rmse"][model_name],  2),
            "NASA Score": round(results["base_nasa"][model_name],  0),
        })
    summary_rows.append({
        "Dataset":    dataset_name,
        "Model":      "Stacking Ensemble",
        "RMSE":       round(results["stacking_rmse"], 2),
        "NASA Score": round(results["stacking_nasa"], 0),
    })

results_table = pd.DataFrame(summary_rows).sort_values(["Dataset", "RMSE"])
results_table

## 6 · SHAP Explanation Agreement

In [ ]:
!pip install -q shap scipy

In [ ]:
import shap
from itertools import combinations
from scipy.stats import spearmanr

RESULTS_DIR = Path("/kaggle/working/cmapss/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = list(get_base_models().keys())
TOP_K_FEATURES = 5   # how many top features to compare between models

print("SHAP version:", shap.__version__)

In [ ]:
# ── Agreement metric functions ────────────────────────────────────────────────

def feature_overlap_score(ranking_a, ranking_b, k=TOP_K_FEATURES):
    """
    What fraction of the top-k features do both models agree on?
    Example: if 3 out of 5 top features match, score = 0.60
    """
    top_features_a = set(ranking_a[:k])
    top_features_b = set(ranking_b[:k])
    shared_features = top_features_a & top_features_b
    return len(shared_features) / k


def exact_rank_match_score(ranking_a, ranking_b, k=TOP_K_FEATURES):
    """
    What fraction of the top-k positions have the exact same feature in both rankings?
    Example: if position 1 and 2 match but 3/4/5 don't, score = 0.40
    """
    matches = sum(ranking_a[i] == ranking_b[i] for i in range(k))
    return matches / k


def pairwise_ordering_score(ranking_a, ranking_b):
    """
    Do both models agree on which feature is more important for every pair of features?
    For example: if both models rank sensor_4 above sensor_7, that counts as agreement.
    """
    position_in_a = {feature: rank for rank, feature in enumerate(ranking_a)}
    position_in_b = {feature: rank for rank, feature in enumerate(ranking_b)}

    all_pairs = list(combinations(ranking_a, 2))
    agreed_pairs = sum(
        (position_in_a[f1] < position_in_a[f2]) == (position_in_b[f1] < position_in_b[f2])
        for f1, f2 in all_pairs
    )
    return agreed_pairs / len(all_pairs) if all_pairs else np.nan


def get_shap_values(trained_model, input_data):
    """
    Calculate SHAP values using TreeExplainer.
    Works with all four tree-based models (Random Forest, XGBoost, LightGBM, CatBoost).
    SHAP values show how much each feature pushed the prediction up or down.
    """
    explainer  = shap.TreeExplainer(trained_model)
    shap_vals  = explainer.shap_values(input_data)

    # Normalise output — different model types return slightly different formats
    if isinstance(shap_vals, list):      shap_vals = shap_vals[0]
    if hasattr(shap_vals, "values"):     shap_vals = shap_vals.values

    return np.asarray(shap_vals)

In [ ]:
def analyze_explanation_agreement(dataset_name, number_of_sample_engines=30):
    """
    Compare how consistently different models explain their predictions.

    Steps:
      1. Load all 4 trained models.
      2. Compute SHAP values for a random sample of test engines.
      3. Rank features by average absolute SHAP value (global importance).
      4. Measure how much the rankings agree between model pairs.
      5. Repeat at the per-engine level to flag low-agreement cases.
    """
    print(f"\n{'='*55}")
    print(f"  Explanation Agreement: {dataset_name}")
    print(f"{'='*55}")

    # Load the feature-engineered test set
    test_data = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_test.csv")
    feature_columns = [
        col for col in test_data.columns
        if col not in ("unit_id", "cycle", "RUL")
    ]

    # Use only the last recorded cycle of each engine
    last_cycle_index = test_data.groupby("unit_id")["cycle"].idxmax()
    X_test_last      = test_data.loc[last_cycle_index, feature_columns].reset_index(drop=True)
    y_test_last      = test_data.loc[last_cycle_index, "RUL"].reset_index(drop=True)

    print(f"Test engines available : {len(X_test_last)}")
    print(f"Feature count          : {len(feature_columns)}")

    # Randomly sample a subset of engines for SHAP (faster computation)
    rng = np.random.RandomState(RANDOM_SEED)
    sample_size    = min(number_of_sample_engines, len(X_test_last))
    sampled_index  = np.sort(rng.choice(len(X_test_last), sample_size, replace=False))
    X_sampled      = X_test_last.iloc[sampled_index].reset_index(drop=True)
    y_sampled      = y_test_last.iloc[sampled_index].reset_index(drop=True)
    print(f"Engines sampled for SHAP: {sample_size}")

    # Load all 4 trained models
    trained_models = {
        model_name: joblib.load(MODEL_DIR / f"{dataset_name}_{model_name}.joblib")
        for model_name in MODEL_NAMES
    }

    # Compute SHAP values for each model
    print("\nComputing SHAP values...")
    shap_values_per_model = {}
    for model_name, trained_model in trained_models.items():
        print(f"  Computing SHAP for {model_name}...")
        shap_values_per_model[model_name] = get_shap_values(trained_model, X_sampled)

    # ── Global agreement (averaged across all sampled engines) ────────────────
    print(f"\n── Global Feature Rankings (top {TOP_K_FEATURES})")

    global_feature_ranking = {}
    for model_name in MODEL_NAMES:
        # Average absolute SHAP across all sampled engines
        mean_shap_per_feature = np.abs(shap_values_per_model[model_name]).mean(axis=0)
        sorted_indices = np.argsort(-mean_shap_per_feature)
        global_feature_ranking[model_name] = [feature_columns[i] for i in sorted_indices]
        print(f"  {model_name}: {global_feature_ranking[model_name][:TOP_K_FEATURES]}")

    # Pairwise comparison between all model pairs
    global_agreement_rows = []
    for model_a, model_b in combinations(MODEL_NAMES, 2):
        ranking_a = global_feature_ranking[model_a]
        ranking_b = global_feature_ranking[model_b]

        # Spearman correlation over all features
        position_a = [global_feature_ranking[model_a].index(f) for f in feature_columns]
        position_b = [global_feature_ranking[model_b].index(f) for f in feature_columns]
        spearman_rho, p_value = spearmanr(position_a, position_b)

        global_agreement_rows.append({
            "Model Pair":              f"{model_a}  vs  {model_b}",
            "Feature Overlap":         feature_overlap_score(ranking_a,   ranking_b),
            "Exact Rank Match":        exact_rank_match_score(ranking_a,  ranking_b),
            "Pairwise Order Agreement":pairwise_ordering_score(ranking_a, ranking_b),
            "Spearman Rho":            round(spearman_rho, 3),
            "p-value":                 round(p_value, 4),
        })

    global_agreement_table = pd.DataFrame(global_agreement_rows)
    print("\nGlobal pairwise agreement:")
    display(global_agreement_table)

    # ── Per-engine agreement ──────────────────────────────────────────────────
    print("\n── Per-Engine Agreement")
    per_engine_scores = []

    for engine_index in range(sample_size):
        # Rank features for this specific engine across all models
        per_engine_ranking = {}
        for model_name in MODEL_NAMES:
            shap_for_engine = shap_values_per_model[model_name][engine_index]
            sorted_indices  = np.argsort(-np.abs(shap_for_engine))
            per_engine_ranking[model_name] = [feature_columns[i] for i in sorted_indices]

        # Average feature overlap across all model pairs for this engine
        overlap_scores = [
            feature_overlap_score(per_engine_ranking[m1], per_engine_ranking[m2])
            for m1, m2 in combinations(MODEL_NAMES, 2)
        ]

        original_engine_id = test_data.loc[last_cycle_index.iloc[sampled_index[engine_index]], "unit_id"]
        per_engine_scores.append({
            "engine_id":          original_engine_id,
            "true_RUL":           y_test_last.iloc[sampled_index[engine_index]],
            "mean_feature_overlap": np.mean(overlap_scores),
            "min_feature_overlap":  np.min(overlap_scores),
        })

    per_engine_table = (
        pd.DataFrame(per_engine_scores)
        .sort_values("mean_feature_overlap")
        .reset_index(drop=True)
    )

    print("\nLowest agreement engines (models disagree most — flag for review):")
    display(per_engine_table.head(5))
    print("\nHighest agreement engines (models agree well):")
    display(per_engine_table.tail(5))

    # Save results
    global_agreement_table.to_csv(RESULTS_DIR / f"{dataset_name}_global_agreement.csv", index=False)
    per_engine_table.to_csv(       RESULTS_DIR / f"{dataset_name}_per_engine_agreement.csv", index=False)

    return global_agreement_table, per_engine_table, global_feature_ranking, shap_values_per_model


agreement_results = {}
for dataset_name in DATASETS:
    agreement_results[dataset_name] = analyze_explanation_agreement(dataset_name)

In [ ]:
# Flag engines where models disagree below this threshold
AGREEMENT_THRESHOLD = 0.40

for dataset_name in DATASETS:
    per_engine_table = agreement_results[dataset_name][1].copy()

    # Engines below the threshold need human review
    per_engine_table["trust_flag"] = np.where(
        per_engine_table["mean_feature_overlap"] < AGREEMENT_THRESHOLD,
        "REVIEW",    # models disagree on which features matter
        "OK"         # models mostly agree
    )

    print(f"\n{dataset_name} — Trust Flags")
    display(per_engine_table.head(10))

## 7 · SHAP-Guided Feature Pruning

In [ ]:
import time
import matplotlib.pyplot as plt

FIGURES_DIR = Path("/kaggle/working/cmapss/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_shap_feature_ranking(dataset_name, sample_size=100, number_of_folds=3):
    """
    Rank features by their SHAP importance using cross-validation on training data only.

    Why cross-validation?
    If we computed SHAP on the full training set using the same model trained on it,
    the importance would be inflated. CV avoids this — SHAP is always computed on
    a fold the model has NOT been trained on. Test data is never used here.
    """
    print(f"\n{'='*55}")
    print(f"  SHAP feature ranking (no leakage): {dataset_name}")
    print(f"{'='*55}")

    training_data = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_train.csv")
    feature_columns = [
        col for col in training_data.columns
        if col not in ("unit_id", "cycle", "RUL")
    ]

    X       = training_data[feature_columns]
    y       = training_data["RUL"]
    engines = training_data["unit_id"]

    print(f"Training rows    : {len(training_data)}")
    print(f"Training engines : {engines.nunique()}")
    print(f"Feature count    : {len(feature_columns)}")

    importance_per_fold = []

    for fold_number, (train_index, val_index) in enumerate(
        GroupKFold(n_splits=number_of_folds).split(X, y, engines), start=1
    ):
        print(f"\n  Fold {fold_number}/{number_of_folds}")

        # Train LightGBM on the training portion of this fold
        fold_model = LGBMRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.08,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
        fold_model.fit(X.iloc[train_index], y.iloc[train_index])

        # Compute SHAP on a random sample from the VALIDATION portion (not test set)
        rng = np.random.RandomState(RANDOM_SEED + fold_number)
        shap_sample_index = rng.choice(len(val_index), min(sample_size, len(val_index)), replace=False)
        X_shap = X.iloc[val_index].iloc[shap_sample_index]

        shap_vals = get_shap_values(fold_model, X_shap)

        # Average absolute SHAP for this fold
        importance_per_fold.append(np.abs(shap_vals).mean(axis=0))
        print(f"  Fold {fold_number} done.")

    # Average importance across all folds
    average_importance = np.mean(importance_per_fold, axis=0)
    sorted_indices     = np.argsort(-average_importance)
    ranked_features    = [feature_columns[i] for i in sorted_indices]

    importance_table = pd.DataFrame({
        "feature":        [feature_columns[i] for i in sorted_indices],
        "mean_abs_shap":  [average_importance[i] for i in sorted_indices],
        "rank":           range(1, len(sorted_indices) + 1),
    })

    importance_table.to_csv(RESULTS_DIR / f"{dataset_name}_shap_importance.csv", index=False)

    print(f"\n  Top 10 features:")
    for i, row in importance_table.head(10).iterrows():
        print(f"    {int(row['rank']):2d}. {row['feature']}  (SHAP={row['mean_abs_shap']:.4f})")

    return ranked_features, feature_columns, importance_table

In [ ]:
def evaluate_feature_pruning(dataset_name, feature_counts_to_try=(5, 10, 15, 20, 25, 30)):
    """
    Train LightGBM with the top-k SHAP-ranked features for several values of k.
    Measure test RMSE, training time, and inference time for each k.
    This shows whether we can use fewer features without sacrificing much accuracy.
    """
    print(f"\n{'='*55}")
    print(f"  Feature pruning tradeoff: {dataset_name}")
    print(f"{'='*55}")

    # Get SHAP-based feature ranking (training data only — no leakage)
    ranked_features, all_features, importance_table = get_shap_feature_ranking(dataset_name)

    # Load data
    training_data = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_train.csv")
    test_data     = pd.read_csv(PROCESSED_DIR / f"features_{dataset_name}_test.csv")

    last_cycle_index = test_data.groupby("unit_id")["cycle"].idxmax()

    # Include the full feature set in the comparison
    feature_counts = sorted(set(list(feature_counts_to_try) + [len(all_features)]))
    feature_counts = [k for k in feature_counts if k <= len(all_features)]

    print(f"\nEvaluating these feature counts: {feature_counts}")

    experiment_rows = []

    for number_of_features in feature_counts:
        # Select top-k features according to the SHAP ranking
        selected_features = ranked_features[:number_of_features]

        X_train = training_data[selected_features]
        y_train = training_data["RUL"]
        X_test  = test_data.loc[last_cycle_index, selected_features]
        y_test  = test_data.loc[last_cycle_index, "RUL"]

        # Train LightGBM with only the selected features
        model = LGBMRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.08,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )

        # Measure training time
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time_seconds = time.time() - start_time

        # Measure inference time
        start_time = time.time()
        predictions = model.predict(X_test)
        inference_time_ms = (time.time() - start_time) / len(X_test) * 1000

        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_test, predictions))

        experiment_rows.append({
            "Number of Features":        number_of_features,
            "Test RMSE":                 round(rmse, 4),
            "Training Time (s)":         round(training_time_seconds, 3),
            "Inference Time (ms/engine)":round(inference_time_ms, 5),
        })

        print(f"  k={number_of_features:3d}  RMSE={rmse:.4f}  "
              f"train={training_time_seconds:.2f}s  infer={inference_time_ms:.4f}ms")

    pruning_results_table = pd.DataFrame(experiment_rows)
    pruning_results_table.to_csv(RESULTS_DIR / f"{dataset_name}_pruning_tradeoff.csv", index=False)

    return pruning_results_table, ranked_features, importance_table


pruning_results = {}
for dataset_name in DATASETS:
    pruning_results[dataset_name] = evaluate_feature_pruning(dataset_name)

In [ ]:
# Plot RMSE and inference time against number of features for each dataset
for dataset_name in DATASETS:
    pruning_table = pruning_results[dataset_name][0]

    fig, (left_plot, right_plot) = plt.subplots(1, 2, figsize=(12, 4))

    # Left: accuracy vs number of features
    left_plot.plot(pruning_table["Number of Features"], pruning_table["Test RMSE"], marker="o")
    left_plot.set_title(f"{dataset_name}: Test RMSE vs Number of Features")
    left_plot.set_xlabel("Number of Features")
    left_plot.set_ylabel("Test RMSE")
    left_plot.grid(alpha=0.3)

    # Right: inference speed vs number of features
    right_plot.plot(pruning_table["Number of Features"], pruning_table["Inference Time (ms/engine)"],
                    marker="s", color="orange")
    right_plot.set_title(f"{dataset_name}: Inference Time vs Number of Features")
    right_plot.set_xlabel("Number of Features")
    right_plot.set_ylabel("Inference Time (ms per engine)")
    right_plot.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{dataset_name}_pruning_tradeoff.png", dpi=150)
    plt.show()

In [ ]:
# Final comparison — feature pruning results for both datasets
for dataset_name in DATASETS:
    print(f"\n{dataset_name} — Pruning Results")
    display(pruning_results[dataset_name][0])